# Day 077 — Exercise 3: should_analyze and save_frame

**What you'll build:** Rate control and frame persistence.

**Why it matters:** Vision LLMs are slow (2-10 s/frame). Analyzing every camera frame at 30 FPS is impossible. `should_analyze` selects frames at a controlled rate. `save_frame` persists frames without any display — `cv2.imshow()` and `plt.show()` both require a display server.

In [ ]:
import numpy as np
from PIL import Image as _PILImage

def _make_mock_frame(h=100, w=100, val=50):
    return np.full((h, w, 3), val, dtype=np.uint8)

class _MockCap:
    def __init__(self, n=5, h=100, w=100):
        self._frames = [_make_mock_frame(h, w) for _ in range(n)]
        self._idx = 0
    def isOpened(self):
        return True
    def read(self):
        if self._idx >= len(self._frames):
            return False, None
        f = self._frames[self._idx]; self._idx += 1
        return True, f
    def release(self):
        pass
    def get(self, prop):
        return 0.0

_mock_camera_fn = lambda device: _MockCap(n=5)
_mock_analyze_fn = lambda img, q: 'FRAME:' + q[:12]
import io, base64

def frame_to_image(frame):
    from PIL import Image
    rgb = frame[:, :, ::-1]
    return Image.fromarray(rgb)

def analyze_frame(frame, question, analyze_fn=None):
    image = frame_to_image(frame)
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']


## Task

1. `should_analyze(frame_count, every_n) -> bool`
   - One line: `return frame_count % every_n == 0`

2. `save_frame(frame, path) -> Path`
   - `out = Path(path)`
   - `frame_to_image(frame).save(out, format='PNG')`
   - `return out`

## Your Implementation

In [ ]:
from pathlib import Path

def should_analyze(frame_count, every_n):
    """Return True if this frame should be analyzed (every_n rate control)."""
    raise NotImplementedError

def save_frame(frame, path):
    """Save a BGR frame to disk as PNG. Returns Path."""
    raise NotImplementedError


In [ ]:
from pathlib import Path

def should_analyze(frame_count, every_n):
    return frame_count % every_n == 0

def save_frame(frame, path):
    out = Path(path)
    frame_to_image(frame).save(out, format='PNG')
    return out


## Automated checks

In [ ]:

score, total = 0, 5
try:
    import tempfile, os

    assert should_analyze(0, 1) is True
    assert should_analyze(1, 1) is True
    score += 1; print("✅ every_n=1: all frames analyzed")

    assert should_analyze(0, 5) is True
    assert should_analyze(5, 5) is True
    assert should_analyze(10, 5) is True
    assert should_analyze(1, 5) is False
    assert should_analyze(4, 5) is False
    score += 1; print("✅ every_n=5: frames 0,5,10,... only")

    assert should_analyze(0, 30) is True
    assert should_analyze(29, 30) is False
    assert should_analyze(30, 30) is True
    score += 1; print("✅ every_n=30 correct")

    frame = _make_mock_frame(50, 50)
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        tmp = f.name
    try:
        result = save_frame(frame, tmp)
        assert str(result) == tmp, f"returned path mismatch: {result}"
        score += 1; print("✅ save_frame returns correct Path")
        assert os.path.getsize(tmp) > 0, "file is empty"
        score += 1; print("✅ save_frame writes non-empty PNG file")
    finally:
        os.unlink(tmp)

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
from pathlib import Path

def should_analyze(frame_count, every_n):
    return frame_count % every_n == 0

def save_frame(frame, path):
    out = Path(path)
    frame_to_image(frame).save(out, format='PNG')
    return out
```

**Why not `cv2.imwrite`?** PIL is already established for all Section 5 image I/O. Using `frame_to_image + PIL.save` keeps the code consistent and avoids any display-related cv2 paths.

</details>